## 定义模型

In [13]:
# from langchain.agents import create_agent
# from langchain.chat_models import init_chat_model
# from dotenv import load_dotenv
# import os
# load_dotenv()

# model = init_chat_model(
#     model="qwen3.5-plus",
#     model_provider="openai",
#     base_url=os.getenv(DASHSCOPE_BASE_URL),
#     api_key=os.getenv(DASHSCOPE_API_KEY)
# )
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from langchain.agents import create_agent

load_dotenv()


model = init_chat_model(
    model="qwen3.5-plus",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

# agent = create_agent(model=model)

## 定义工具

In [14]:
from langchain_tavily import TavilySearch
from langchain_core.tools import tool

tavily=TavilySearch(
    max_results=5,
    topic="general"
)
@tool
def web_search(query: str):
    """根据关键词搜索互联网"""
    return tavily.invoke(query)

## 添加记忆管理

In [15]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

connection = sqlite3.connect("resources/personal_chef.db", check_same_thread=False)
checkpointer= SqliteSaver(connection)
checkpointer.setup()

## 定义智能体

In [16]:
from langchain.agents import create_agent

system_prompt="""
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：
1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份 “当前可用食材清单”。
2.智能食谱检索：优先调用 web_search 工具，以 “可用食材清单” 为核心关键词，查找可行菜谱。
3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由、食谱的参考图片，帮助用户快速做出决策。

请严格按照流程，优先调用 web_search 工具搜索食谱，搜索不到的情况下才能自己发挥。
"""
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=checkpointer
)

## 测试

In [17]:
from langchain_core.messages import HumanMessage

multimodel_messages= HumanMessage([
    {
        "type": "text", "text": "帮我看看能做什么"
    },
    {
        "type": "image", "url": "https://pic.rmb.bdstatic.com/bjh/bc117c78c118/250321/3e2503fe8b412a0e7c470d9cf828f4d5.jpeg"
    }
])

config = {"configurable": {"thread_id": "5"}}
response = agent.invoke({
    "messages": [multimodel_messages]
}, config)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么'}, {'type': 'image', 'url': 'https://pic.rmb.bdstatic.com/bjh/bc117c78c118/250321/3e2503fe8b412a0e7c470d9cf828f4d5.jpeg'}]
================================== Ai Message ==================================
Tool Calls:
  web_search (call_dd3d03918df445b5afb52442)
 Call ID: call_dd3d03918df445b5afb52442
  Args:
    query: 葡萄 苹果 李子 橙子 鸡蛋 小番茄 能做什么菜谱
================================= Tool Message =================================
Name: web_search

{"query": "葡萄 苹果 李子 橙子 鸡蛋 小番茄 能做什么菜谱", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.reddit.com/r/EatCheapAndHealthy/comments/xzxcc4/i_have_a_bunch_of_cherry_tomatoes_what_do_i_do/?tl=zh-hans", "title": "我有一堆小番茄，怎么办啊？有什么好吃的做法推荐吗？ - Reddit", "content": "把它们扔进橄榄油里，加盐和胡椒，再加点蒜，然后在300度烤到它们开始起泡。可以做成cottage cheese、牛油果吐司或意面的配料。", "score": 0.4391465, "raw_content": null}, {"url": "https://www.yout